# Network clock — module-size sweep: random-CpG vs PageRank-central CpG (+ Fig 6 comparison)

This notebook repeats the **Ridge network-clock sweep** (same split, same reps, same
`RidgeCV` fitter as `05_network_clock_ridge.ipynb`) and compares **two ways of choosing the
single representative CpG for each module**, then overlays the **published Network Clock** curve
from the composite Fig 6 for reference:

1. **Random CpG per module** — pick one CpG at random from each module (the CpG itself if
   the module is a singleton). Stochastic → run `N_REPS_SWEEP` times per `N`, plot mean ± std.
2. **PageRank-central CpG per module** — pick the module's single most central CpG by
   **PageRank** on the module's induced subgraph of the τ=0.70 network (the WGCNA "hub" idea,
   matching the original top-modules clock). Deterministic → run **once** per `N`.
3. **Network Clock (Fig 6)** — the random-1-per-module ridge sweep that feeds panel (a) of the
   composite figure, **loaded** from its saved CSV (not re-run). This is the *same method* as
   sweep 1 above, so the two should overlap — it is plotted here as a reproducibility check.

In all variants modules are taken **in order of size (largest first)**, the first `N` modules
selected, and **RidgeCV** fitted on the resulting `samples × N` CpG matrix.

We sweep `N` over the same grid as the ridge clock and plot **test R² and MAE vs N**, using
the same colours and plotting aesthetics as `test_network_clock_3cohorts.ipynb`.

> *Naming*: we say **modules**, never "clusters".

In [ ]:
import os, warnings, pickle, json
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import networkx as nx
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, median_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")

# ── Paths (match 05_network_clock_ridge.ipynb) ──────────────────────
FILTERED_BETAS = "outputs/01_filtered_betas/BetaMatrix_0.35.tsv"
SAMPLESHEET    = "outputs/01_filtered_betas/Samplesheet.csv"
MODULE_CSV     = "outputs/02_network/module_assignments_0.70.csv"
NETWORK_GEXF   = "outputs/02_network/network_0.70.gexf"   # needed for PageRank

OUT_DIR = Path("outputs/05_clock_random_vs_pagerank")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Sweep parameters — IDENTICAL to the ridge clock ─────────────────
MODULE_COUNTS_TO_TEST = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200, 300, 400, 500]
OLS_MAX_M        = 1500          # guard cap (ridge is fine for p >= n, kept for parity)
N_REPS_SWEEP     = 100           # reps per N for the RANDOM variant only
RANDOM_BASE_SEED = 12345         # base seed for the random method (same as ridge clock)
TEST_FRACTION    = 0.2
SEED             = 42

# Ridge hyperparameters (same family as the ridge clock's RidgeCV)
RIDGE_ALPHAS   = np.logspace(-3, 3, 25)
RIDGE_CV_FOLDS = 5

# ── Reference clocks (drawn as baselines, from the cached summary if present) ──
REFERENCE_CLOCKS = ["horvath2013", "hannum", "altumage", "skinandblood", "han"]
CLOCK_LABELS = {
    "horvath2013": "Horvath", "hannum": "Hannum", "altumage": "AltumAge",
    "skinandblood": "Skin & Blood", "han": "Han",
}
REF_SUMMARY_CSV = Path("outputs/05_clock/all_modes_summary.csv")  # optional

# ── Published Network Clock sweep (the random-1-per-module curve in Fig 6 (a)) ──
# Loaded, not re-run. First existing path wins. Method name was 'random_per_cluster'
# in the main pipeline (aliased to 'random_per_module' in some copies) — both accepted.
FIG6_SWEEP_CANDIDATES = [
    Path("outputs/05_clock/random/random_sweep_results.csv"),
    Path("outputs/05_clock_ridge/random/random_sweep_results.csv"),
    Path("../outputs/05_clock/random/random_sweep_results.csv"),
]
FIG6_COLOR = "#4477AA"   # blue, dashed — distinct from the new orange/green curves
FIG6_LABEL = "Network Clock — Fig. 6 (random 1/module)"

# ── Style — matches test_network_clock_3cohorts.ipynb ───────────────
DPI = 300; FRAME_LW = 2.0
mpl.rcParams.update({
    "font.family": ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 16, "axes.labelsize": 16, "axes.titlesize": 16,
    "xtick.labelsize": 16, "ytick.labelsize": 16,
    "legend.fontsize": 13, "legend.frameon": False,
    "axes.edgecolor": "black", "axes.linewidth": FRAME_LW,
    "axes.spines.top": True, "axes.spines.right": True,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.major.size": 6, "ytick.major.size": 6,
    "xtick.major.width": 1.2, "ytick.major.width": 1.2,
    "figure.dpi": DPI, "savefig.dpi": DPI, "savefig.bbox": "tight",
    "figure.facecolor": "white", "axes.facecolor": "white",
})

# Method colours — consistent with the ridge clock family in the test notebook
METHOD_COLOR = {
    "random_per_module_ridge": "#EE9933",   # orange — random 1 per module (Ridge)
    "pagerank_ridge":          "#117733",   # green  — PageRank-central per module (Ridge)
}
METHOD_LABEL = {
    "random_per_module_ridge": "Random 1 per module (size-ordered)",
    "pagerank_ridge":          "PageRank-central per module (Ridge)",
}
REF_COLOR_MAP = {"horvath2013": "#999933", "hannum": "#CC6677",
                 "altumage": "#117733", "skinandblood": "#882255",
                 "han": "#88CCEE"}
STUDY_PALETTE = ["#4477AA","#EE6677","#228833","#CCBB44","#66CCEE",
                 "#AA3377","#BBBBBB","#E69F00","#56B4E9","#009E73",
                 "#D55E00","#CC79A7","#0072B2","#F0E442","#882255"]


def _save(fig, path):
    p = Path(path)
    fig.savefig(str(p), dpi=DPI)
    fig.savefig(str(p.with_suffix(".svg")))
    print(f"  -> {p}")


## Load betas, ages, module assignments

In [ ]:
def load_methylation(beta_path, sheet_path):
    """Sample × CpG betas + aligned metadata (Age, Study_ID). Mirrors the loader
    used across the project so the same inputs work unchanged."""
    print(f"Loading {beta_path} ...")
    df = pd.read_csv(beta_path, sep="\t", low_memory=False)
    cpg_cols = [c for c in df.columns if str(c).startswith("cg")]
    if not cpg_cols:
        fc = df.columns[0]
        if df[fc].astype(str).head(20).str.startswith("cg").sum() > 10:
            df = df.set_index(fc).T.reset_index()
            df.rename(columns={df.columns[0]: "Sample_ID"}, inplace=True)
        cpg_cols = [c for c in df.columns if str(c).startswith("cg")]
    idc = next((c for c in ["Sample_ID","ID_REF","ID","Sample","geo_accession"]
                if c in df.columns), None)
    df_meth = df[cpg_cols].apply(pd.to_numeric, errors="coerce")
    if idc: df_meth.index = df[idc].astype(str)

    meta = pd.read_csv(sheet_path)
    for c in meta.columns:
        if c.lower() == "age": meta = meta.rename(columns={c: "Age"})
    meta["Age"] = pd.to_numeric(meta["Age"], errors="coerce")
    mid = next((c for c in ["Sample_ID","ID_REF","ID","Sample","geo_accession"]
                if c in meta.columns), None)
    if mid: meta = meta.set_index(meta[mid].astype(str))
    common = df_meth.index.intersection(meta.index)
    df_meth, meta = df_meth.loc[common], meta.loc[common]
    if "Study_ID" not in meta.columns:
        for alt in ["GSE","Study","Series","Dataset"]:
            if alt in meta.columns: meta = meta.rename(columns={alt:"Study_ID"}); break
        if "Study_ID" not in meta.columns: meta["Study_ID"] = "unknown"
    print(f"  Loaded: {df_meth.shape[0]} samples × {df_meth.shape[1]:,} CpGs")
    return df_meth, meta


df_meth, meta = load_methylation(FILTERED_BETAS, SAMPLESHEET)
ages      = meta["Age"].values
study_ids = meta["Study_ID"].values

module_dict = pd.read_csv(MODULE_CSV).set_index("CpG")["Module"].to_dict()

# Group CpGs by module (only CpGs present in the filtered matrix)
module_cpgs = defaultdict(list)
for cpg, m in module_dict.items():
    if cpg in df_meth.columns:
        module_cpgs[m].append(cpg)

# Order modules by SIZE, largest first (ties broken by module id for determinism)
modules_by_size = sorted(module_cpgs.keys(),
                         key=lambda m: (-len(module_cpgs[m]), m))
sizes = [len(module_cpgs[m]) for m in modules_by_size]
n_multi = sum(1 for s in sizes if s > 1)
print(f"  Modules: {len(modules_by_size):,}  "
      f"(multi-CpG: {n_multi}, singletons: {len(modules_by_size)-n_multi})")
print(f"  Largest 5 module sizes: {sizes[:5]}")
print(f"  Total CpGs in modules: {sum(sizes):,}")


## Train/test split (same seed as the ridge clock)

In [ ]:
idx = np.arange(len(ages))
i_tr, i_te = train_test_split(idx, test_size=TEST_FRACTION, random_state=SEED)
print(f"  Train: n={len(i_tr)}  |  Test: n={len(i_te)}")

# Counts to actually sweep (cap + can't exceed number of modules)
test_counts = sorted(set(n for n in MODULE_COUNTS_TO_TEST
                         if n <= OLS_MAX_M and n <= len(modules_by_size)))
print(f"  Sweep N values: {test_counts}")


## The shared Ridge fitter + the random feature builder

`fit_ridge_with_split` is the same fit used by `05_network_clock_ridge.ipynb`: standardise the
selected CpG matrix on the training fold, fit `RidgeCV`, report held-out R²/MAE. Both new
variants pick exactly one CpG per module; they differ only in *which* CpG.

In [ ]:
def fit_ridge_with_split(X_tr, X_te, y_tr, y_te, s_tr=None, s_te=None):
    """Standardise on train, fit RidgeCV, return metrics + predictions. Same
    recipe as the ridge network clock."""
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)
    m = RidgeCV(alphas=RIDGE_ALPHAS, cv=RIDGE_CV_FOLDS)
    m.fit(X_tr_s, y_tr)
    p_tr, p_te = m.predict(X_tr_s), m.predict(X_te_s)
    return {
        "r2_train": r2_score(y_tr, p_tr), "r2_test": r2_score(y_te, p_te),
        "mae_train": mean_absolute_error(y_tr, p_tr),
        "mae_test":  mean_absolute_error(y_te, p_te),
        "medae_test": median_absolute_error(y_te, p_te),
        "alpha": float(m.alpha_), "n_features": X_tr.shape[1],
        "y_test": y_te, "pred_test": p_te, "studies_test": s_te,
    }


def build_features_random(selected_modules, rng):
    """One randomly chosen CpG per module (the CpG itself if singleton).
    Returns a (samples × N) DataFrame aligned to df_meth.index."""
    chosen = []
    for m in selected_modules:
        cpgs = module_cpgs[m]
        chosen.append(cpgs[0] if len(cpgs) == 1 else rng.choice(cpgs))
    return df_meth[chosen].copy()


## Sweep 1 — random CpG per module (Ridge), `N_REPS_SWEEP` reps per N

In [ ]:
y = np.asarray(ages, dtype=float)
s = np.asarray(study_ids)
y_tr, y_te = y[i_tr], y[i_te]
s_tr, s_te = s[i_tr], s[i_te]

rand_rows = []
rand_best_by_N = {}

print(f"Random-per-module sweep — {N_REPS_SWEEP} reps per N")
for N in test_counts:
    selected = modules_by_size[:N]            # N largest modules
    r2s, maes = [], []
    best_r2, best_res = -np.inf, None
    for rep in range(N_REPS_SWEEP):
        rng = np.random.RandomState(RANDOM_BASE_SEED + N * 1000 + rep)
        X = build_features_random(selected, rng)
        X = X.fillna(X.mean())
        res = fit_ridge_with_split(X.values[i_tr], X.values[i_te],
                                   y_tr, y_te, s_tr, s_te)
        r2s.append(res["r2_test"]); maes.append(res["mae_test"])
        rand_rows.append({"method": "random_per_module_ridge", "N": N, "rep": rep,
                          "r2_test": res["r2_test"], "mae_test": res["mae_test"]})
        if res["r2_test"] > best_r2:
            best_r2, best_res = res["r2_test"], res
    rand_best_by_N[N] = best_res
    print(f"  N={N:4d}  R²_test={np.mean(r2s):.3f}±{np.std(r2s):.3f}  "
          f"MAE_test={np.mean(maes):.2f}±{np.std(maes):.2f}")

df_rand = pd.DataFrame(rand_rows)
df_rand.to_csv(OUT_DIR / "sweep_random_per_module.csv", index=False)
print(f"\n  Saved → {OUT_DIR / 'sweep_random_per_module.csv'}")


## Sweep 2 — PageRank-central CpG per module (Ridge)

Each module is represented by its **single most central CpG**, chosen by **PageRank** on the
module's induced subgraph of the τ=0.70 network. Deterministic given the graph → run **once
per N**. The representative CpG for every module is precomputed once (independent of N).

Requires `network_0.70.gexf`. If missing, the code falls back to degree centrality, then to a
deterministic first-CpG pick, and warns.

In [ ]:
def _load_network(path):
    try:
        G = nx.read_gexf(path)
        print(f"  Loaded network: {G.number_of_nodes():,} nodes, "
              f"{G.number_of_edges():,} edges ← {path}")
        return G
    except FileNotFoundError:
        print(f"  ⚠ {path} not found — PageRank cannot use graph connectivity.")
        return None
    except Exception as e:
        print(f"  ⚠ Could not read {path} ({type(e).__name__}: {e}).")
        return None


def compute_pagerank_reps(modules, G):
    """For each module, pick the representative CpG = highest PageRank within the
    module's induced subgraph. Falls back to highest degree, then to the first
    CpG (deterministic) if the graph is unavailable or a module has no edges.

    Returns {module_id: chosen_cpg}.
    """
    reps = {}
    n_pr, n_deg, n_first = 0, 0, 0
    for m in modules:
        cpgs = module_cpgs[m]
        if len(cpgs) == 1:
            reps[m] = cpgs[0]; n_first += 1
            continue
        if G is not None:
            present = [c for c in cpgs if c in G]
            sub = G.subgraph(present) if present else None
            if sub is not None and sub.number_of_edges() > 0:
                pr = nx.pagerank(sub, alpha=0.85, weight="weight")
                reps[m] = max(sorted(pr), key=lambda c: pr[c]); n_pr += 1   # tie-break by id
                continue
            elif present:
                deg = dict(G.degree(present))
                reps[m] = max(sorted(deg), key=lambda c: deg[c]); n_deg += 1
                continue
        reps[m] = sorted(cpgs)[0]; n_first += 1
    print(f"  Representative CpGs: {n_pr} by PageRank, {n_deg} by degree, "
          f"{n_first} singletons/fallback")
    return reps


G_net = _load_network(NETWORK_GEXF)
pagerank_reps_all = compute_pagerank_reps(modules_by_size, G_net)


In [ ]:
pagerank_rows = []
pagerank_by_N = {}

print("PageRank-central sweep — deterministic, 1 run per N")
for N in test_counts:
    selected = modules_by_size[:N]
    chosen = [pagerank_reps_all[m] for m in selected]
    X = df_meth[chosen].copy()
    X = X.fillna(X.mean())
    res = fit_ridge_with_split(X.values[i_tr], X.values[i_te],
                               y_tr, y_te, s_tr, s_te)
    pagerank_by_N[N] = res
    pagerank_rows.append({"method": "pagerank_ridge", "N": N,
                          "r2_test": res["r2_test"], "mae_test": res["mae_test"],
                          "r2_train": res["r2_train"], "mae_train": res["mae_train"],
                          "alpha": res["alpha"]})
    print(f"  N={N:4d}  R²_test={res['r2_test']:.3f}  MAE_test={res['mae_test']:.2f}  "
          f"alpha={res['alpha']:.3g}")

df_pagerank = pd.DataFrame(pagerank_rows)
df_pagerank.to_csv(OUT_DIR / "sweep_pagerank.csv", index=False)
print(f"\n  Saved → {OUT_DIR / 'sweep_pagerank.csv'}")

best_N_pagerank = int(df_pagerank.sort_values("r2_test", ascending=False).iloc[0]["N"])
best_N_rand     = max(rand_best_by_N, key=lambda k: rand_best_by_N[k]["r2_test"])
print(f"\n  Best PageRank: N={best_N_pagerank}  R²={pagerank_by_N[best_N_pagerank]['r2_test']:.3f}")
print(f"  Best random:   N={best_N_rand}  R²={rand_best_by_N[best_N_rand]['r2_test']:.3f}")


## Reference-clock baselines (optional, from cached summary)

In [ ]:
ref_pooled = {}
if REF_SUMMARY_CSV.exists():
    df_sum = pd.read_csv(REF_SUMMARY_CSV)
    if "mode" in df_sum.columns:
        df_sum = df_sum[df_sum["mode"] == "random"]
    pretty_to_key = {v: k for k, v in CLOCK_LABELS.items()}
    for _, row in df_sum.iterrows():
        cl = str(row.get("clock", ""))
        key = cl if cl in REFERENCE_CLOCKS else pretty_to_key.get(cl)
        if key is None:
            for pretty, k in pretty_to_key.items():
                if cl.startswith(pretty): key = k; break
        if key in REFERENCE_CLOCKS and "r2_test" in row and pd.notna(row["r2_test"]):
            ref_pooled.setdefault(key, float(row["r2_test"]))
    print(f"  Reference clock R² (test) from cache:")
    for k, v in ref_pooled.items():
        print(f"    {CLOCK_LABELS[k]:13s} {v:+.3f}")
else:
    print(f"  {REF_SUMMARY_CSV} not found — baselines will be omitted from the plot.")


## Load the published Network Clock sweep (Fig 6 comparison)

Loads the random-1-per-module ridge sweep that feeds panel (a) of the composite Fig 6 — **not
re-run**, just read from its saved CSV. Same method as Sweep 1, so the curves should overlap;
the overlay is a reproducibility check. If the CSV isn't found, the comparison is skipped and
the rest of the notebook still runs.

In [ ]:
def load_fig6_network_clock():
    """Return a tidy DataFrame [N, r2_test, mae_test] for the random-1-per-module
    ridge sweep used in Fig 6, or None if not found."""
    src = next((p for p in FIG6_SWEEP_CANDIDATES if p.exists()), None)
    if src is None:
        print("  ⚠ Fig-6 sweep CSV not found in any candidate location — "
              "skipping the published Network Clock curve.")
        return None
    df = pd.read_csv(src)
    if "method" in df.columns:
        mask = df["method"].astype(str).str.contains(
            "random_per_cluster|random_per_module", case=False, regex=True)
        df = df[mask]
    if df.empty:
        print(f"  ⚠ No random-per-module rows in {src} — skipping Fig-6 curve.")
        return None
    ncol = next((c for c in ["n_modules", "N", "n_cpgs", "M"] if c in df.columns), None)
    if ncol is None:
        print(f"  ⚠ No N/n_modules column in {src} — skipping Fig-6 curve.")
        return None
    out = df[[ncol, "r2_test", "mae_test"]].rename(columns={ncol: "N"})
    print(f"  Loaded Fig-6 Network Clock sweep ← {src}  "
          f"({out['N'].nunique()} N values, {len(out)} rows)")
    return out


df_fig6 = load_fig6_network_clock()


## R² and MAE vs N — random vs PageRank vs Fig 6

Same plotting aesthetic as `test_network_clock_3cohorts.ipynb`: in-tick frames, the method
colours, the published Fig-6 curve as a dashed blue line, reference clocks as dashed baselines
with leader-line labels on the right.

In [ ]:
def _agg(df, key, ncol="N"):
    g = df.groupby(ncol)[key].agg(["mean", "std"]).reset_index()
    return g[ncol].values, g["mean"].values, g["std"].fillna(0).values


def draw_baselines_with_arrows(ax, x_data_max, ref_rows,
                                seg_x_frac=0.82, label_x_frac=1.03,
                                xlim_pad_frac=1.20, y_label_band=(0.10, 0.92),
                                label_fontsize=12):
    """Dashed baseline segments on the right + sign-staggered leader labels."""
    seg_x0, seg_x1 = seg_x_frac * x_data_max, x_data_max
    label_x = label_x_frac * x_data_max
    ax.set_xlim(0, xlim_pad_frac * x_data_max)
    for key, val in ref_rows:
        ax.hlines(val, seg_x0, seg_x1, colors=REF_COLOR_MAP[key],
                  linestyles="--", lw=1.3, alpha=0.95)
    rows = sorted(ref_rows, key=lambda r: -r[1])
    ymin, ymax = ax.get_ylim()
    ys = np.linspace(ymin + (ymax-ymin)*y_label_band[1],
                     ymin + (ymax-ymin)*y_label_band[0], len(rows))
    for (key, val), ylab in zip(rows, ys):
        ax.annotate(CLOCK_LABELS[key], xy=(seg_x1, val), xycoords="data",
                    xytext=(label_x, ylab), textcoords="data",
                    fontsize=label_fontsize, fontweight="bold",
                    color=REF_COLOR_MAP[key], va="center", ha="left",
                    arrowprops=dict(arrowstyle="-", color=REF_COLOR_MAP[key],
                                    lw=0.9, shrinkA=1, shrinkB=1),
                    annotation_clip=False)


fig, axes = plt.subplots(1, 2, figsize=(15, 6.2))

# ── (a) R² vs N ─────────────────────────────────────────────
ax = axes[0]
xr, mr, sr = _agg(df_rand, "r2_test")
ax.errorbar(xr, mr, yerr=sr, fmt="o-", color=METHOD_COLOR["random_per_module_ridge"],
            lw=2.0, markersize=7, capsize=3, label=METHOD_LABEL["random_per_module_ridge"])
xp, mp, sp = _agg(df_pagerank, "r2_test")
ax.errorbar(xp, mp, yerr=sp, fmt="D-", color=METHOD_COLOR["pagerank_ridge"],
            lw=2.0, markersize=6, capsize=3, label=METHOD_LABEL["pagerank_ridge"])
if df_fig6 is not None:
    xf, mf, sf = _agg(df_fig6, "r2_test")
    ax.errorbar(xf, mf, yerr=sf, fmt="s--", color=FIG6_COLOR,
                lw=1.8, markersize=6, capsize=3, alpha=0.9, label=FIG6_LABEL)
ax.set_ylim(0.15, 0.95)
if ref_pooled:
    draw_baselines_with_arrows(ax, float(max(test_counts)),
                               [(k, v) for k, v in ref_pooled.items()])
ax.set_xlabel("Number of CpGs (N)"); ax.set_ylabel(r"Test $R^{2}$")
ax.legend(loc="lower right", handlelength=2.0, fontsize=11)
ax.set_box_aspect(1.0)

# ── (b) MAE vs N ────────────────────────────────────────────
ax = axes[1]
xr, mr, sr = _agg(df_rand, "mae_test")
ax.errorbar(xr, mr, yerr=sr, fmt="o-", color=METHOD_COLOR["random_per_module_ridge"],
            lw=2.0, markersize=7, capsize=3, label=METHOD_LABEL["random_per_module_ridge"])
xp, mp, sp = _agg(df_pagerank, "mae_test")
ax.errorbar(xp, mp, yerr=sp, fmt="D-", color=METHOD_COLOR["pagerank_ridge"],
            lw=2.0, markersize=6, capsize=3, label=METHOD_LABEL["pagerank_ridge"])
if df_fig6 is not None:
    xf, mf, sf = _agg(df_fig6, "mae_test")
    ax.errorbar(xf, mf, yerr=sf, fmt="s--", color=FIG6_COLOR,
                lw=1.8, markersize=6, capsize=3, alpha=0.9, label=FIG6_LABEL)
ax.set_xlabel("Number of CpGs (N)"); ax.set_ylabel("Test MAE (years)")
ax.legend(loc="upper right", handlelength=2.0, fontsize=11)
ax.set_box_aspect(1.0)

fig.tight_layout()
_save(fig, OUT_DIR / "random_vs_pagerank_vs_fig6_sweep")
plt.show()


## Predicted vs actual at the best N (random + PageRank)

The Fig-6 sweep CSV stores only per-N metrics (not per-sample predictions), so the scatter
shows the two re-run variants at their best N.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6.2))
for ax, (title, res) in zip(axes, [
        (f"Random 1/module (Ridge), N={best_N_rand}", rand_best_by_N[best_N_rand]),
        (f"PageRank-central/module (Ridge), N={best_N_pagerank}", pagerank_by_N[best_N_pagerank])]):
    yv, pv, sv = res["y_test"], res["pred_test"], res["studies_test"]
    if sv is not None and len(set(sv)) > 1:
        for i, st in enumerate(sorted(set(sv))):
            mask = np.array(sv) == st
            ax.scatter(yv[mask], pv[mask], s=28, alpha=0.75,
                       color=STUDY_PALETTE[i % len(STUDY_PALETTE)],
                       edgecolor="white", linewidth=0.3, label=st)
    else:
        ax.scatter(yv, pv, s=28, alpha=0.75, color="#666", edgecolor="white", linewidth=0.3)
    mlin, blin = np.polyfit(yv, pv, 1)
    xl = np.linspace(yv.min(), yv.max(), 100)
    ax.plot(xl, mlin*xl + blin, "-", color="black", lw=2.2, zorder=4)
    r2 = r2_score(yv, pv); mae = mean_absolute_error(yv, pv)
    ax.set_xlabel("Chronological age (years)"); ax.set_ylabel("Predicted age (years)")
    ax.set_title(f"{title}\n$R^2$={r2:.3f}, MAE={mae:.2f}y")
    ax.set_box_aspect(1.0)
fig.tight_layout()
_save(fig, OUT_DIR / "scatter_best_N_both")
plt.show()


## Combined results table (random + PageRank + Fig 6)

In [ ]:
# Build over the union of N values so nothing is lost if Fig-6 used a different grid.
all_N = sorted(set(test_counts) |
               (set(int(n) for n in df_fig6["N"].unique()) if df_fig6 is not None else set()))

rows = []
for N in all_N:
    row = {"N": N}
    sub = df_rand[df_rand["N"] == N]
    if len(sub):
        row["random_r2_mean"]  = sub["r2_test"].mean()
        row["random_r2_std"]   = sub["r2_test"].std()
        row["random_mae_mean"] = sub["mae_test"].mean()
        row["random_mae_std"]  = sub["mae_test"].std()
    if N in pagerank_by_N:
        p = pagerank_by_N[N]
        row["pagerank_r2"]  = p["r2_test"]
        row["pagerank_mae"] = p["mae_test"]
    if df_fig6 is not None:
        f = df_fig6[df_fig6["N"] == N]
        if len(f):
            row["fig6_r2_mean"]  = f["r2_test"].mean()
            row["fig6_r2_std"]   = f["r2_test"].std()
            row["fig6_mae_mean"] = f["mae_test"].mean()
            row["fig6_mae_std"]  = f["mae_test"].std()
    rows.append(row)

df_combined = pd.DataFrame(rows)
df_combined.to_csv(OUT_DIR / "combined_summary.csv", index=False)
print(df_combined.round(3).to_string(index=False))
print(f"\n  Saved → {OUT_DIR / 'combined_summary.csv'}")
